# Experiment 2: Analyzing Weather Data from OpenWeatherMap API

Get a free API key at https://openweathermap.org/api. Then either add it in Colab (key icon in left sidebar -> Secrets, name `OWM_API_KEY`) or paste it when prompted below.

In [ ]:
%pip install -q folium

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# API key: from Colab Secrets if available, otherwise ask for it
try:
    from google.colab import userdata
    api_key = userdata.get("OWM_API_KEY")
except Exception:
    from getpass import getpass
    api_key = getpass("Enter your OpenWeatherMap API key: ")

city = "Pune"
url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}&units=metric"

In [ ]:
response = requests.get(url)
data = response.json()
print(data)

weather = {
    "City": data["name"],
    "Temperature": data["main"]["temp"],
    "Feels Like": data["main"]["feels_like"],
    "Humidity": data["main"]["humidity"],
    "Pressure": data["main"]["pressure"],
    "Weather": data["weather"][0]["description"],
    "Wind Speed": data["wind"]["speed"],
}
df = pd.DataFrame([weather])
display(df)

In [ ]:
print("City:", weather["City"])
print("Temperature:", weather["Temperature"], "°C")
print("Humidity:", weather["Humidity"], "%")
print("Pressure:", weather["Pressure"], "hPa")
print("Wind Speed:", weather["Wind Speed"], "m/s")
print("Weather:", weather["Weather"])

In [ ]:
values = [weather["Temperature"], weather["Feels Like"], weather["Humidity"], weather["Wind Speed"]]
labels = ["Temperature", "Feels Like", "Humidity", "Wind Speed"]

plt.bar(labels, values)
plt.title("Weather Analysis - " + city)
plt.ylabel("Values")
plt.xticks(rotation=20)
plt.show()

df.to_csv("weather_data.csv", index=False)
print("Weather data saved successfully!")

## Extended analysis: 5-day forecast data (trends, aggregation, correlation)

In [ ]:
forecast_url = f"https://api.openweathermap.org/data/2.5/forecast?q={city}&appid={api_key}&units=metric"
fdata = requests.get(forecast_url).json()

rows = []
for item in fdata["list"]:
    rows.append({
        "Datetime": item["dt_txt"],
        "Temperature": item["main"]["temp"],
        "Humidity": item["main"]["humidity"],
        "Pressure": item["main"]["pressure"],
        "Wind Speed": item["wind"]["speed"],
        "Precipitation": item.get("rain", {}).get("3h", 0),
    })
fdf = pd.DataFrame(rows)

# Cleaning
fdf["Datetime"] = pd.to_datetime(fdf["Datetime"])
fdf = fdf.drop_duplicates().dropna().set_index("Datetime")
display(fdf.head())
print(fdf.describe())

In [ ]:
# Trend + moving average (window=7 readings)
fdf["Moving_Avg"] = fdf["Temperature"].rolling(window=7).mean()
plt.figure(figsize=(10, 4))
plt.plot(fdf["Temperature"], label="Temperature")
plt.plot(fdf["Moving_Avg"], label="Moving Average")
plt.title("Temperature Trend - " + city)
plt.legend(); plt.grid(True); plt.show()

# Daily aggregation
daily = fdf[["Temperature", "Humidity", "Wind Speed", "Precipitation"]].resample("D").mean()
display(daily)
daily["Temperature"].plot(kind="bar", title="Average Daily Temperature")
plt.ylabel("°C"); plt.show()

# Correlation (Pearson) + heatmap
corr = fdf[["Temperature", "Humidity", "Pressure", "Wind Speed"]].corr()
print("Temperature vs Humidity:", fdf["Temperature"].corr(fdf["Humidity"]))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar()
plt.xticks(range(len(corr)), corr.columns, rotation=30)
plt.yticks(range(len(corr)), corr.columns)
plt.title("Correlation Heatmap")
plt.show()

fdf.to_csv("weather_forecast_data.csv")

In [ ]:
# Geospatial visualization with folium
import folium
lat, lon = data["coord"]["lat"], data["coord"]["lon"]
m = folium.Map(location=[lat, lon], zoom_start=10)
folium.Marker([lat, lon],
              popup=f"{city}: {weather['Temperature']}°C, {weather['Weather']}").add_to(m)
m